# Calculate regional mortality with parametric bootstrapping

This may require large memory ~30GB.

In [ ]:
import os
import xarray as xr
import warnings
from utils.utils import get_scenario_config
from utils.mortality_utils import att_frac

In [ ]:
# === Path config ===
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/region/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
TMREL_DIR = "/glade/derecho/scratch/awells/air_quality/TMREL/"
BETA_DIR = "/glade/derecho/scratch/awells/air_quality/beta_ozone/"
BMR_DIR = "/glade/derecho/scratch/awells/air_quality/BMR_ozone//"

In [ ]:
# Load country masks
mask_file = "GBD_Region_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

# Load population file
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)
pop = population.reindex_like(masks, method="nearest", tolerance=1e-9)

In [ ]:
# === Calculate the scalar distributions ===
n_samples = 200

# TMREL from GBD21 (uniform distribution)
tmrel_file = f"TMREL_{n_samples}_samples_ozone.nc"
tmrel_path = os.path.join(TMREL_DIR, tmrel_file)
tmrel_da = xr.open_dataarray(tmrel_path)

# Beta from RR per 10ppb (normal distribution)
beta_file = f"beta_{n_samples}_samples_ozone.nc"
beta_path = os.path.join(BETA_DIR, beta_file)
beta_da = xr.open_dataarray(beta_path)

# Load BMR for each grid point (normal distribution)
bmr_file = f"GBD_BMR_Country_Mask_COPD_{n_samples}_samples_1990-2009.nc"
bmr_path = os.path.join(BMR_DIR, bmr_file)
BMR = xr.open_dataarray(bmr_path)  # three quantiles

In [ ]:
warnings.filterwarnings('ignore')

# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

O3_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/mortality/ozone/region/{n_samples}_samples/"

for ens_num in ensemble_members:
    print(f"Processing ensemble member {ens_num:02d}")
    # {years.stop - 1} from OSDMA8 calculation
    dates = f"{years.start}-{years.stop - 1}"

    # Load ozone data
    o3_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    o3_path = os.path.join(O3_DIR, o3_file)
    o3 = xr.open_dataarray(o3_path).astype("float32")
    o3 = o3.reindex_like(masks, method="nearest", tolerance=1e-9, fill_value=0)

    del o3_file, o3_path

    # make o3 dask-backed
    o3 = o3.chunk({'lat': 180, 'lon': 360})

    # range doesn't include the final year which is okay
    # because OSDMA8 excludes final year
    for year in years:
        print(f"Processing year {year}")
        o3_year = o3.sel(year=year)
        # Calculation the attributable fraction
        AF = att_frac(o3_year, tmrel_da, beta_da).chunk({"samples": 10})

        POP = pop.sel(year=year).chunk({"lat": 180, "lon": 360})
        # Calculate mortality at each grid point for n samples
        M = AF * BMR * POP

        del o3_year, AF, POP

        # Calculate regional mortality - save per year per region
        for region in masks.region:
            print(f"Processing {region.data}")

            region_mask = masks.sel(region=region)
            region_M = M.where(region_mask == 1).sum(dim=("lat", "lon"))
            region_M = region_M.expand_dims(region=[region.data])

            del region_mask

            description = ("Regional mortality (COPD) due to ozone "
                           f"sample size {n_samples} - scripts by A.F."
                           " Wells (2025)")
            region_M.attrs["description"] = description
            region_M.attrs["model"] = model
            region_M.attrs["scenario"] = scenario
            region_M.attrs["ensemble_number"] = ens_num
            region_M.attrs["region"] = region.data
            region_M.attrs["year"] = year

            out_file = f"Regional_mortality_{n_samples}samples_{model}_{scenario}_{ens_num:02d}_{region.data}_{year}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)
            print(f"Saving to {out_path}")
            region_M.to_netcdf(out_path)

            del region_M

        del M

    del o3

print("All processing complete.")